## Complex Machine Learning Models and Keras Part 2

#### This script follows the structure below

## 1.Importing libraries
## 2.Data Wrangling
## 3.Reshape for running the Model
## 4.Data Split
## 5.Random Forest Model
## 6.Unconvering Feature Importance
## 7.Top Three Weather Station - Features Importance Analysis

# 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import operator
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn import datasets  
from sklearn.ensemble import RandomForestClassifier
from numpy import argmax
from sklearn.model_selection import train_test_split
from sklearn import metrics  
from sklearn.tree import plot_tree
from sklearn import tree

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Creating a path for importing the climate data set

path = r'/Users/daniel/Desktop/Ordner/Data Analyst/Data Analytics Course/Data Specialization/Data Sets'

In [ ]:
# Import the data set

df_weather = pd.read_csv(os.path.join(path, 'weather_clean.csv'),index_col = False)

df_answer = pd.read_csv(os.path.join(path, 'Dataset-Answers-Weather_Prediction_Pleasant_Weather.csv'),index_col = False)

# 2. Data Wrangling

In [ ]:
# Show all the columns in the data set
pd.set_option('display.max_columns', 200)

# Show all the rows in the data set
pd.set_option('display.max_rows', None)

#### Dataset Weather

In [ ]:
# Check for correct import 

df_weather.head()

In [ ]:
# Check for shape

df_weather.shape

In [ ]:
# Check for missing values

df_weather.isnull().sum()

In [ ]:
# Check for duplicate values

duplicate_value = df_weather.duplicated()

print(f"Total duplicate value: {duplicate_value.sum()}")

In [ ]:
# Check the basic statistic values
df_weather.describe()

#### Dataset Answers

In [ ]:
# Check for correct import 

df_answer.head()

In [ ]:
# Check for the shape

df_answer.shape

In [ ]:
# Check for missing values

df_answer.isnull().sum()

In [ ]:
# Check for duplicate values

duplicate_value = df_answer.duplicated()

print(f"Total duplicate value: {duplicate_value.sum()}")

#### Reduce data to one decade. Chosen decade: 2010 - 2019

In [ ]:
# Reduce observations dataset to 2010's

df_decade = df_weather[(df_weather['DATE'].astype(str).str[:4] >= '2010') & (df_weather['DATE'].astype(str).str[:4] <= '2019')]
df_decade

In [ ]:
# Reduce answers dataset to 2010's

answers_decade = df_answer[(df_answer['DATE'].astype(str).str[:4] >= '2010') & (df_answer['DATE'].astype(str).str[:4] <= '2019')]
answers_decade

In [ ]:
# Extract stations list

stations = [col.split('_')[0] for col in df_decade.columns if '_' in col]

In [ ]:
# Create a set of unique station names

unique_stations = set(stations)
unique_stations

In [ ]:
# Create a dictionary to store the frequency of entries for each station
station_frequencies = {}

for station in unique_stations:
    # Select columns that belong to the current station
    station_columns = [col for col in  df_decade.columns if col.startswith(station)]
    
    # Count non-missing entries across all columns for the station
    station_frequencies[station] =  df_decade[station_columns].notna().sum().sum()

# Print the frequency of entries for each station
print("Frequency of entries for each weather station:")
for station, freq in station_frequencies.items():
    print(f"{station}: {freq} entries")

In [ ]:
# Drop unnecessary columns

df_decade.drop(['DATE', 'MONTH'], axis=1, inplace=True)
df_decade.head()

In [ ]:
df_decade.shape # observations dataset has the correct shape

In [ ]:
answers_decade.drop(columns = 'DATE', inplace = True)

In [ ]:
answers_decade.shape # predictions dataset has the correct shape

# 3. Reshaping for running the Model

#### The final shapes should be X = (3652, 135) and y = (3652,) for one decade of information.

In [ ]:
X = df_decade

In [ ]:
y = answers_decade

In [ ]:
# Turn X and y from a df to arrays

X = np.array(X)
y = np.array(y)

In [ ]:
X.shape

In [ ]:
y.shape

# 4. Data Split

In [ ]:
# Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X,y,random_state = 42)

In [ ]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

# 5.Random Forest Model

In [ ]:
# Create a RF classifier
clf = RandomForestClassifier(n_estimators = 100)#, max_depth=5)  
  
# Training the model on the training dataset
# fit function is used to train the model using the training sets as parameters
clf.fit(X_train, y_train)

In [ ]:
# Perform predictions on the test dataset
y_pred = clf.predict(X_test)
  
# Use metrics module for accuracy calculation
print("Model Accuracy: ", metrics.accuracy_score(y_test, y_pred))

In [ ]:
# Class-names = {0:'Unpleasant Weather', 1:'Pleasant Weather'}

fig = plt.figure(figsize=(80,40))
plot_tree(clf.estimators_[15], fontsize = 20, filled=True);

# 6. Uncovering Feature Importances

In [ ]:
# Retrieve feature importances from the trained model

newarray = clf.feature_importances_
print(clf.feature_importances_.shape)
newarray

In [ ]:
# Reshape newarray

newarray = newarray.reshape(-1,15,9)
print(newarray.shape)
newarray

In [ ]:
# Collapse this shape into one observation for each weather station

sumarray = np.sum(newarray[0], axis=1)
sumarray

In [ ]:
# Convert the set of unique stations to a list

unique_stations_list = list(unique_stations)

In [ ]:
important = pd.Series(sumarray, index = unique_stations_list)
important = important.sort_values(ascending = False)
important

In [ ]:
# Create a df to associate weather stations with their importances

df_importance = pd.DataFrame({
    'Weather Station': unique_stations_list,
    'Importance': sumarray
})

df_importance = df_importance.sort_values(by='Importance', ascending = False)

In [ ]:
# Plot the results

%matplotlib inline

plt.style.use('fivethirtyeight')
print(unique_stations_list)

plt.bar(df_importance['Weather Station'], df_importance['Importance'], orientation = 'vertical')
plt.xticks(rotation='vertical')
plt.xlabel('Weather Station')
plt.ylabel('Importance')
plt.title('Weather Station Importance 2010s')

plt.savefig(os.path.join(path, 'Visualizations', 'Station_feature_importances.png'), bbox_inches='tight')

plt.show()

# 7. Top Three Weather Station - Features Importance Analysis

## 7.1 HEATHROW

In [ ]:
# Create a list of the columns containing "MUNCHENB" in their names
HEATHROW_list = [col for col in df_weather.columns if 'HEATHROW' in col]
HEATHROW_list

In [ ]:
# Create a dataframe with those columns

df_HEATHROW = df_weather[HEATHROW_list]
df_HEATHROW

In [ ]:
# Reduce answers dataset to Basels answers only

answers_HEATHROW = df_answer['HEATHROW_pleasant_weather']
answers_HEATHROW

In [ ]:
df_HEATHROW.shape # observations dataset has the correct shape

In [ ]:
answers_HEATHROW.shape # predictions dataset has the correct shape

### Reshaping for modeling

In [ ]:
X2 = df_HEATHROW

In [ ]:
y2 = answers_HEATHROW

In [ ]:
# Turn X2 and y2 from df to arrays

X = np.array(X2)
y = np.array(y2)

In [ ]:
X.shape

In [ ]:
y.shape

### Data Split

In [ ]:
# Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X,y,random_state = 42)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
X_test

In [ ]:
y_test

### Random Forest Model Conduction

In [ ]:
# Create a RF classifier
clf = RandomForestClassifier(n_estimators = 100)#, max_depth=5)  
  
# Training the model on the training dataset
# fit function is used to train the model using the training sets as parameters
clf.fit(X_train, y_train)

In [ ]:
# Perform predictions on the test dataset
y_pred = clf.predict(X_test)
  
# Use metrics module for accuracy calculation
print("Model Accuracy: ", metrics.accuracy_score(y_test, y_pred))

In [ ]:
# Define class names
class_names = {0: 'Unpleasant Weather', 1: 'Pleasant Weather'}

# Plot the 7th decision tree in the random forest
fig1 = plt.figure(figsize=(20, 10))  # Reduced from (80, 40) for practical display
plot_tree(clf.estimators_[6], 
          fontsize=12, 
          filled=True, 
          class_names=[class_names[i] for i in clf.estimators_[6].classes_])

plt.savefig(os.path.join(path, 'Visualizations', 'HEATHROW_FOREST.png'), bbox_inches='tight')

plt.show()

### Uncovering Feature Importance

In [ ]:
# Retrieve feature importances from the trained model

newarray = clf.feature_importances_
print(clf.feature_importances_.shape)
newarray

In [ ]:
# Create a list of weather features

wx_list = [feature.replace('HEATHROW_', '') for feature in HEATHROW_list]
wx_list

In [ ]:
important = pd.Series(newarray, index = wx_list)
important

In [ ]:
plt.style.use('fivethirtyeight')

# List of x locations for plotting
x_values = list(range(len(newarray)))

# Debugging output (optional)
print(wx_list)

# Create the figure and plot
fig2 = plt.figure(figsize=(12, 6))  # Add a figure object with a reasonable size
plt.bar(x_values, newarray, orientation='vertical')
plt.xticks(x_values, wx_list, rotation='vertical')
plt.ylabel('Importance')
plt.xlabel('Feature')
plt.title('Feature Importances for HEATHROW (all years)')

# Save the figure
plt.savefig(os.path.join(path, 'Visualizations', 'HEATHROW_feature_importances.png'), bbox_inches='tight')
plt.show()


## 7.2 BUDAPEST

In [ ]:
# Create a list of the columns containing "MUNCHENB" in their names
BUDAPEST_list = [col for col in df_weather.columns if 'BUDAPEST' in col]
BUDAPEST_list

In [ ]:
# Create a dataframe with those columns

df_BUDAPEST = df_weather[BUDAPEST_list]
df_BUDAPEST

In [ ]:
# Reduce answers dataset to Basels answers only

answers_BUDAPEST = df_answer['BUDAPEST_pleasant_weather']
answers_BUDAPEST

In [ ]:
df_BUDAPEST.shape # observations dataset has the correct shape

In [ ]:
answers_BUDAPEST.shape # predictions dataset has the correct shape

### Reshaping for modeling

In [ ]:
X3 = df_BUDAPEST

In [ ]:
y3 = answers_BUDAPEST

In [ ]:
# Turn X2 and y2 from df to arrays

X = np.array(X3)
y = np.array(y3)

In [ ]:
X.shape

In [ ]:
y.shape

### Data Split

In [ ]:
# Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X,y,random_state = 42)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
X_test

In [ ]:
y_test

### Random Forest Model Conduction

In [ ]:
# Create a RF classifier
clf = RandomForestClassifier(n_estimators = 100)#, max_depth=5)  
  
# Training the model on the training dataset
# fit function is used to train the model using the training sets as parameters
clf.fit(X_train, y_train)

In [ ]:
# Perform predictions on the test dataset
y_pred = clf.predict(X_test)
  
# Use metrics module for accuracy calculation
print("Model Accuracy: ", metrics.accuracy_score(y_test, y_pred))

In [ ]:
# Define class names
class_names = {0: 'Unpleasant Weather', 1: 'Pleasant Weather'}

# Plot the 7th decision tree in the random forest
fig3 = plt.figure(figsize=(20, 10))  # Reduced from (80, 40) for practical display
plot_tree(clf.estimators_[6], 
          fontsize=12, 
          filled=True, 
          class_names=[class_names[i] for i in clf.estimators_[6].classes_])

plt.savefig(os.path.join(path, 'Visualizations', 'BUDAPEST_FOREST.png'), bbox_inches='tight')

plt.show()


### Uncovering Feature Importance

In [ ]:
# Retrieve feature importances from the trained model

newarray = clf.feature_importances_
print(clf.feature_importances_.shape)
newarray

In [ ]:
# Create a list of weather features

wx_list = [feature.replace('BUDAPEST_', '') for feature in BUDAPEST_list]
wx_list

In [ ]:
important = pd.Series(newarray, index = wx_list)
important

In [ ]:
plt.style.use('fivethirtyeight')

# List of x locations for plotting
x_values = list(range(len(newarray)))

# Debugging output (optional)
print(wx_list)

# Create the figure and plot
fig4 = plt.figure(figsize=(12, 6))  # Add a figure object with a reasonable size
plt.bar(x_values, newarray, orientation='vertical')
plt.xticks(x_values, wx_list, rotation='vertical')
plt.ylabel('Importance')
plt.xlabel('Feature')
plt.title('Feature Importances for BUDAPEST (all years)')

# Save the figure
plt.savefig(os.path.join(path, 'Visualizations', 'BUDAPEST_feature_importances.png'), bbox_inches='tight')
plt.show()

## 7.3 BELGRADE

In [ ]:
# Create a list of the columns containing "MUNCHENB" in their names
BELGRADE_list = [col for col in df_weather.columns if 'BELGRADE' in col]
BELGRADE_list

In [ ]:
# Create a dataframe with those columns

df_BELGRADE = df_weather[BELGRADE_list]
df_BELGRADE

In [ ]:
# Reduce answers dataset to Basels answers only

answers_BELGRADE = df_answer['BELGRADE_pleasant_weather']
answers_BELGRADE

In [ ]:
df_BELGRADE.shape # observations dataset has the correct shape

In [ ]:
answers_BELGRADE.shape # predictions dataset has the correct shape

### Reshaping for modeling

In [ ]:
X4 = df_BELGRADE

In [ ]:
y4 = answers_BELGRADE

In [ ]:
# Turn X2 and y2 from df to arrays

X = np.array(X4)
y = np.array(y4)

In [ ]:
X.shape

In [ ]:
y.shape

### Data Split

In [ ]:
# Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X,y,random_state = 42)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
X_test

In [ ]:
y_test

### Random Forest Model Conduction

In [ ]:
# Create a RF classifier
clf = RandomForestClassifier(n_estimators = 100)#, max_depth=5)  
  
# Training the model on the training dataset
# fit function is used to train the model using the training sets as parameters
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)
  
# Use metrics module for accuracy calculation
print("Model Accuracy: ", metrics.accuracy_score(y_test, y_pred))

In [ ]:
# Define class names
class_names = {0: 'Unpleasant Weather', 1: 'Pleasant Weather'}

# Plot the 7th decision tree in the random forest
fig5 = plt.figure(figsize=(20, 10))  # Reduced from (80, 40) for practical display
plot_tree(clf.estimators_[6], 
          fontsize=12, 
          filled=True, 
          class_names=[class_names[i] for i in clf.estimators_[6].classes_])

plt.savefig(os.path.join(path, 'Visualizations', 'BELGRADE_FOREST.png'), bbox_inches='tight')

plt.show()

### Uncovering Feature Importance

In [ ]:
# Retrieve feature importances from the trained model

newarray = clf.feature_importances_
print(clf.feature_importances_.shape)
newarray

In [ ]:
# Create a list of weather features

wx_list = [feature.replace('BELGRADE_', '') for feature in BELGRADE_list]
wx_list

In [ ]:
important = pd.Series(newarray, index = wx_list)
important

In [ ]:
plt.style.use('fivethirtyeight')

# List of x locations for plotting
x_values = list(range(len(newarray)))

# Debugging output (optional)
print(wx_list)

# Create the figure and plot
fig4 = plt.figure(figsize=(12, 6))  # Add a figure object with a reasonable size
plt.bar(x_values, newarray, orientation='vertical')
plt.xticks(x_values, wx_list, rotation='vertical')
plt.ylabel('Importance')
plt.xlabel('Feature')
plt.title('Feature Importances for BELGRADE (all years)')

# Save the figure
plt.savefig(os.path.join(path, 'Visualizations', 'BELGRADE_feature_importances.png'), bbox_inches='tight')
plt.show()